# 🎧 Huấn Luyện Lại Model Gợi Ý Âm Nhạc Embeat 45M (Google Colab)

Notebook này được tối ưu để chạy độc lập (**100% Standalone trên Google Colab**). Bạn không cần phải tải cả thư mục dự án lên Google Drive.

### Tính năng chính:
1. **Stream & Filter an toàn bộ nhớ (Safe Disk Stream):** Dùng `hf_hub_download` tải từng phân vùng Parquet xuống ổ cứng Colab (~80 GB đĩa) rồi đọc theo Row Groups, **chống sập RAM (OOM)**.
2. **Bộ Lọc Chất Lượng Cao (Khắc phục triệt để lỗi gợi ý không mượt):**
   - Bắt buộc có nhãn thể loại `artist_genres` (loại bỏ hoàn toàn 37% bài khuyết thể loại).
   - Hỗ trợ cả chế độ lấy Toàn Bộ (`TARGET_TRACKS = None`, `MIN_POPULARITY = 0`) hoặc lấy theo giới hạn mong muốn.
   - Làm sạch và chuẩn hóa 9 đặc trưng âm học.
3. **Huấn luyện ma trận âm học & thể loại thưa (Multi-Hot CSR):**
4. **Tự động xuất và tải về:** Đóng gói đúng định dạng artifact `embeat_recommender.joblib` tương thích 100% với Backend của dự án.

## Bước 1: Cài đặt thư viện cần thiết

In [ ]:
!pip install -q pyarrow pandas scikit-learn scipy joblib requests huggingface_hub
print("✓ Đã cài đặt xong các thư viện cần thiết!")

## Bước 2: Khai báo cấu trúc Model EmbeatRecommender chuẩn

In [ ]:
from typing import Any, Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, normalize
import joblib

AUDIO_FEATURES = [
    "danceability", "energy", "valence", "tempo",
    "acousticness", "instrumentalness", "speechiness", "liveness", "loudness"
]

class EmbeatRecommender:
    def __init__(self):
        self.df = None
        self.scaler = None
        self.feature_matrix = None
        self.id_to_idx = {}
        self.genre_matrix = None

    def fit(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        self.scaler = StandardScaler()
        self.feature_matrix = self.scaler.fit_transform(self.df[AUDIO_FEATURES])
        self.id_to_idx = {tid: idx for idx, tid in enumerate(self.df["track_id"])}

        genre_list = self.df["artist_genres"].fillna("").apply(
            lambda x: [g.strip().lower() for g in str(x).split(",") if g.strip()]
        )
        mlb = MultiLabelBinarizer(sparse_output=True)
        self.genre_matrix = csr_matrix(mlb.fit_transform(genre_list))
        return self

print("✓ Class EmbeatRecommender đã sẵn sàng.")

## Bước 3: Cấu hình Hugging Face Token & Thu thập dữ liệu sạch

> **Lưu ý quan trọng:**
> - Truy cập https://huggingface.co/datasets/GD-Studio/embeat_45m_spotify_tracks và bấm **"Agree and access repository"** (hoặc Accept License).
> - Lấy Access Token tại https://huggingface.co/settings/tokens (quyền Read).
> - **TARGET_TRACKS = None**: Lấy toàn bộ dataset (không giới hạn).
> - **TARGET_TRACKS = 500000** hoặc **1000000**: Khuyên dùng cho Colab Free để tốc độ nhanh nhất.

In [ ]:
import os
import gc
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

# ------------------------------------------------------------------
# 1. ĐIỀN TOKEN CỦA BẠN VÀO DƯỚI ĐÂY:
HF_TOKEN = "hf_your_access_token_here"
# ------------------------------------------------------------------

# 2. CẤU HÌNH SỐ LƯỢNG BÀI:
# - Muốn lấy toàn bộ: đặt TARGET_TRACKS = None và MIN_POPULARITY = 0
# - Khuyên dùng an toàn cho Colab: TARGET_TRACKS = 500000 hoặc 1000000
TARGET_TRACKS = None     # None = Lấy toàn bộ; hoặc đặt số ví dụ: 500000, 1000000
MIN_POPULARITY = 0       # 0 = Lấy toàn bộ độ phổ biến; hoặc 15 để lọc bớt bài rác

PARQUET_FILES = [
    "data/train-00000-of-00003.parquet",
    "data/train-00001-of-00003.parquet",
    "data/train-00002-of-00003.parquet",
]

collected_dfs = []
total_collected = 0

target_label = f"{TARGET_TRACKS:,}" if TARGET_TRACKS is not None else "Toàn bộ dataset"
print(f"Bắt đầu stream và lọc dữ liệu từ Embeat 45M (Mục tiêu: {target_label})...")

for file_idx, rel_path in enumerate(PARQUET_FILES):
    if TARGET_TRACKS is not None and total_collected >= TARGET_TRACKS:
        break
    print(f"\n[Phân vùng {file_idx+1}/3] Đang tải {rel_path} xuống đĩa Colab...")
    try:
        local_file = hf_hub_download(
            repo_id="GD-Studio/embeat_45m_spotify_tracks",
            filename=rel_path,
            repo_type="dataset",
            token=HF_TOKEN.strip(),
        )
    except Exception as exc:
        print(f"❌ Lỗi tải phân vùng {rel_path}: {exc}")
        print("👉 Vui lòng kiểm tra lại HF_TOKEN và đảm bảo bạn đã bấm 'Accept License' tại trang dataset.")
        break
        
    pfile = pq.ParquetFile(local_file)
    num_groups = pfile.num_row_groups
    print(f"✓ Đã kết nối! File có {num_groups} row groups. Đang đọc và lọc...")

    for rg in range(num_groups):
        table = pfile.read_row_group(rg)
        batch_df = table.to_pandas()
        
        # Lọc dữ liệu sạch và chuẩn hóa:
        # 1. Bắt buộc có artist_genres hợp lệ
        # 2. Popularity >= MIN_POPULARITY
        # 3. Đầy đủ 9 chỉ số âm học
        cleaned = batch_df.dropna(subset=AUDIO_FEATURES + ["track_id", "artist_genres"])
        cleaned = cleaned[cleaned["tempo"] > 0]
        if MIN_POPULARITY > 0:
            cleaned = cleaned[cleaned["popularity"] >= MIN_POPULARITY]
        cleaned = cleaned[cleaned["artist_genres"].astype(str).str.strip() != ""]
        
        if not cleaned.empty:
            collected_dfs.append(cleaned)
            total_collected += len(cleaned)
            print(f"  -> Đã thu thập: {total_collected:,} / {target_label} bài", end="\r")
            
        if TARGET_TRACKS is not None and total_collected >= TARGET_TRACKS:
            break
            
    gc.collect()

if collected_dfs:
    print(f"\nĐang hợp nhất và khử trùng lặp dữ liệu...")
    final_df = pd.concat(collected_dfs, ignore_index=True).drop_duplicates(subset=["track_id"])
    if TARGET_TRACKS is not None:
        final_df = final_df.head(TARGET_TRACKS)
    del collected_dfs
    gc.collect()
    print(f"✓ Hoàn tất thu thập! Tổng số bài sạch trong model: {len(final_df):,} bài hát.")
    display(final_df[["track_name", "artist_name", "artist_genres", "popularity"]].head(5))
else:
    print("❌ Không thu thập được bài hát nào. Hãy kiểm tra lại HF_TOKEN.")

## Bước 4: Huấn luyện và Đóng gói Artifact

In [ ]:
print("Đang huấn luyện EmbeatRecommender...")
model = EmbeatRecommender()
model.fit(final_df)

print("\n✓ HUẤN LUYỆN THÀNH CÔNG!")
print(f" - Tổng số bài hát: {len(model.df):,}")
print(f" - Ma trận âm học: {model.feature_matrix.shape}")
print(f" - Ma trận thể loại: {model.genre_matrix.shape} ({model.genre_matrix.shape[1]:,} dòng nhạc phân biệt)")

# Đóng gói theo chuẩn dictionary {'version': 1, 'model': model} của Backend
artifact_payload = {
    "version": 1,
    "model": model
}

output_file = "embeat_recommender.joblib"
joblib.dump(artifact_payload, output_file, compress=3)
print(f"✓ Đã đóng gói thành công file: {output_file}")

## Bước 5: Tải file Model mới về máy tính của bạn

In [ ]:
from google.colab import files
print("Đang mở hộp thoại tải file embeat_recommender.joblib về máy...")
files.download("embeat_recommender.joblib")